# Lunar Router — End-to-End Test

Tests the full flow: push API key → route request → verify ClickHouse trace.

**Prerequisites:**
- ClickHouse running on `localhost:8123/9000`
- Go engine running on `localhost:8080`
- Python API running on `localhost:8000`
- Mistral key saved via the UI Data Sources page

In [ ]:
import httpx
import json

API_URL = "http://localhost:8000"
ENGINE_URL = "http://localhost:8080"
CH_URL = "http://localhost:8123"

## 1. Check services are running

In [ ]:
# Python API
r = httpx.get(f"{API_URL}/health")
print("Python API:", r.json())

# Go Engine
r = httpx.get(f"{ENGINE_URL}/health")
print("Go Engine:", r.json())

# ClickHouse
r = httpx.get(f"{CH_URL}/", params={"query": "SELECT 1", "user": "default", "password": "lunar"})
print("ClickHouse:", r.text.strip())

## 2. Verify Mistral key is configured

In [ ]:
r = httpx.get(f"{API_URL}/v1/secrets")
secrets = r.json()
print("Configured providers:", secrets["configured_providers"])
assert "mistral" in secrets["configured_providers"], "Mistral key not found! Add it via the UI Data Sources page."

## 3. Push stored keys to Go engine

In [ ]:
from lunar_router.storage.secrets import push_all_to_engine

push_all_to_engine()

# Verify engine has the key
r = httpx.get(f"{ENGINE_URL}/v1/config/keys")
print("Engine configured providers:", r.json())

## 4. Send a chat request to Mistral via the Go engine

In [ ]:
r = httpx.post(
    f"{ENGINE_URL}/v1/chat/completions",
    json={
        "model": "mistral-small-latest",
        "messages": [
            {"role": "user", "content": "What is 2+2? Reply in one word."}
        ],
        "max_tokens": 10,
        "temperature": 0,
    },
    timeout=30.0,
)

print(f"Status: {r.status_code}")
resp = r.json()
print(f"Model: {resp.get('model')}")
print(f"Response: {resp['choices'][0]['message']['content']}")
if resp.get('usage'):
    print(f"Tokens: {resp['usage']}")
if resp.get('cost'):
    print(f"Cost: ${resp['cost']['total_cost_usd']:.6f}")

## 5. Verify trace was written to ClickHouse

In [ ]:
import time
time.sleep(6)  # Wait for async batch flush (5s interval)

r = httpx.get(
    f"{CH_URL}/",
    params={
        "query": "SELECT selected_model, provider, latency_ms, tokens_in, tokens_out, total_cost_usd, is_error FROM llm_traces ORDER BY timestamp DESC LIMIT 5 FORMAT JSON",
        "user": "default",
        "password": "lunar",
        "database": "lunar_router",
    },
)

data = r.json()
print(f"Total traces: {data['rows']}")
for row in data["data"]:
    print(f"  {row['selected_model']} via {row['provider']} — {row['latency_ms']:.0f}ms, {row['tokens_in']}+{row['tokens_out']} tokens, ${row['total_cost_usd']:.6f}")

## 6. Verify trace appears in the UI analytics endpoint

In [ ]:
r = httpx.get(f"{API_URL}/v1/stats/default/analytics", params={"days": 1, "trace_limit": 5})
analytics = r.json()

print(f"Total requests: {analytics['totals']['request_count']}")
print(f"Total cost: ${analytics['totals']['total_cost_usd']:.6f}")
print(f"Avg latency: {analytics['totals']['avg_latency_s']:.3f}s")
print(f"Success rate: {analytics['totals']['success_rate']}%")
print(f"\nModels:")
for m in analytics['series']['by_model']:
    print(f"  {m['model_id']}: {m['request_count']} requests, ${m['total_cost_usd']:.6f}")
print(f"\nLatest traces:")
for t in analytics['raw_sample'][:3]:
    status = '✓' if t['is_success'] else '✗'
    print(f"  {status} {t['model_id']} — {t['latency_s']:.3f}s, {t['input_tokens']}+{t['output_tokens']} tokens")

## 7. Send multiple requests to test routing traces

In [ ]:
prompts = [
    "Explain quantum computing in one sentence.",
    "Write a Python function to reverse a string.",
    "What is the capital of France?",
    "Translate 'hello world' to Japanese.",
    "What are the prime numbers below 20?",
]

for prompt in prompts:
    r = httpx.post(
        f"{ENGINE_URL}/v1/chat/completions",
        json={
            "model": "mistral-small-latest",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 50,
            "temperature": 0,
        },
        timeout=30.0,
    )
    resp = r.json()
    answer = resp["choices"][0]["message"]["content"][:60]
    cost = resp.get("cost", {}).get("total_cost_usd", 0)
    print(f"  [{r.status_code}] ${cost:.6f} — {answer}...")

print(f"\nSent {len(prompts)} requests. Check the UI Traces page!")

## 8. Final check — traces in ClickHouse after batch flush

In [ ]:
time.sleep(6)

r = httpx.get(
    f"{CH_URL}/",
    params={
        "query": "SELECT count() as total, sum(total_cost_usd) as cost, avg(latency_ms) as avg_lat FROM llm_traces WHERE selected_model LIKE '%mistral%' FORMAT JSON",
        "user": "default",
        "password": "lunar",
        "database": "lunar_router",
    },
)

data = r.json()["data"][0]
print(f"Mistral traces: {data['total']}")
print(f"Total cost: ${float(data['cost']):.6f}")
print(f"Avg latency: {float(data['avg_lat']):.0f}ms")
print("\n✅ End-to-end test complete! Open http://localhost:3000/traces to see them in the UI.")